# Teslimat Gecikmesi — EDA + Baseline

**Hedef:** `Late_delivery_risk` (0/1)

Akış: veriyi yükle → keşfet → leakage temizliği → Logistic Regression baseline.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    ConfusionMatrixDisplay,
)

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))
from src.config import TARGET, DATA_RAW

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

## 1) Veriyi yükle

`data/raw/` altına Kaggle CSV'sini koy. Tipik dosya adı: `DataCoSupplyChainDataset.csv`

In [ ]:
candidates = list(DATA_RAW.glob("*.csv"))
if not candidates:
    raise FileNotFoundError(
        f"CSV bulunamadı: {DATA_RAW}\n"
        "Önce README'deki Kaggle indirme adımlarını tamamla."
    )

# En büyük CSV genelde ana tablo
csv_path = max(candidates, key=lambda p: p.stat().st_size)
print("Kullanılan dosya:", csv_path.name)

df = pd.read_csv(csv_path, encoding="latin-1", low_memory=False)
print("Şekil:", df.shape)
df.head()

## 2) EDA — hızlı bakış

In [ ]:
print("Kolonlar:")
print(df.columns.tolist())
print("\nEksik değer (ilk 20):")
missing = df.isna().mean().sort_values(ascending=False)
display(missing.head(20))

if TARGET not in df.columns:
    raise KeyError(f"Hedef kolon yok: {TARGET}")

print("\nSınıf dengesi:")
print(df[TARGET].value_counts(normalize=True).round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
df[TARGET].value_counts().sort_index().plot(kind="bar", ax=ax, color=["#4C78A8", "#F58518"])
ax.set_title("Late_delivery_risk dağılımı")
ax.set_xlabel("Sınıf")
ax.set_ylabel("Adet")
plt.tight_layout()
plt.show()

## 3) Temizlik — leakage ve gereksiz ID'ler

Gerçek teslim sonucunu sızdıran kolonları çıkarıyoruz.

In [ ]:
leaky = [
    "Days for shipping (real)",
    "Delivery Status",
]
id_like = [
    "Order Id",
    "Order Customer Id",
    "Customer Id",
    "Product Card Id",
    "Product Category Id",
    "Order Item Cardprod Id",
    "Order Item Id",
    "Category Id",
    "Customer Email",
    "Customer Password",
    "Product Description",
    "Product Image",
    "Customer Fname",
    "Customer Lname",
    "Customer Street",
    "Order Zipcode",
    "Product Status",
]

drop_cols = [c for c in leaky + id_like if c in df.columns]
work = df.drop(columns=drop_cols).copy()
print("Düşürülen kolon sayısı:", len(drop_cols))
print("Kalan şekil:", work.shape)

## 4) Baseline — Logistic Regression

İlk sürüm: sayısal + kategorik (düşük kardinalite) özelliklerle hızlı skor.

In [ ]:
y = work[TARGET].astype(int)
X = work.drop(columns=[TARGET])

# Tarih kolonlarını şimdilik çıkar (sonra feature engineering)
date_cols = [c for c in X.columns if "date" in c.lower() or "Date" in c]
X = X.drop(columns=date_cols, errors="ignore")

num_cols = X.select_dtypes(include=["number"]).columns.tolist()
cat_cols = [c for c in X.select_dtypes(exclude=["number"]).columns if X[c].nunique() <= 40]
high_card = [c for c in X.select_dtypes(exclude=["number"]).columns if c not in cat_cols]
X = X.drop(columns=high_card)

print("Sayısal:", len(num_cols), "| Kategorik:", len(cat_cols), "| Yüksek kardinalite atıldı:", len(high_card))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
    ]
)

clf = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

clf.fit(X_train, y_train)
proba = clf.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print(classification_report(y_test, pred, digits=3))
print("ROC-AUC:", round(roc_auc_score(y_test, proba), 4))

ConfusionMatrixDisplay.from_predictions(y_test, pred)
plt.title("Confusion Matrix — Logistic Regression")
plt.show()

## Sonraki adımlar

1. Tarihlerden özellik üret (`order month`, `shipping days scheduled` vb.)
2. Random Forest + XGBoost
3. Cross-validation ve hiperparametre araması
4. Feature importance yorumu